In [275]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import re



In [ ]:
name = 'csv/test021_NCH_spi.csv'
df = pd.read_csv(name, sep=';')



In [305]:
# Verifica cómo quedó
print(df.head(41))


              Time  UART CH1
0,000135945  Start       NaN
0,000140285   0x00       NaN
0,000175005   Stop       NaN
0,000179335  Start       NaN
0,000183675   0x00       NaN
0,000218395   Stop       NaN
0,000222705  Start       NaN
0,000227045   0x30       NaN
0,000261765   Stop       NaN
0,000266095  Start       NaN
0,000270435   0x00       NaN
0,000305155   Stop       NaN
0,000309475  Start       NaN
0,000313815   0x00       NaN
0,000348535   Stop       NaN
0,000352860  Start       NaN
0,000357200   0x31       NaN
0,000391920   Stop       NaN
0,000518775  Start       NaN
0,000523115   0x00       NaN
0,000557835   Stop       NaN
0,000562155  Start       NaN
0,000566495   0x00       NaN
0,000601215   Stop       NaN
0,000605540  Start       NaN
0,000609880   0x32       NaN
0,000644600   Stop       NaN
0,000648915  Start       NaN
0,000653255   0x00       NaN
0,000687975   Stop       NaN
0,000692310  Start       NaN
0,000696650   0x00       NaN
0,000731370   Stop       NaN
0,000735690  S

In [306]:

df = df.drop('UART CH1', axis = 1)
df["tiempos"]  = df.index
# Verifica cómo quedó
print(df.head(20))

              Time      tiempos
0,000135945  Start  0,000135945
0,000140285   0x00  0,000140285
0,000175005   Stop  0,000175005
0,000179335  Start  0,000179335
0,000183675   0x00  0,000183675
0,000218395   Stop  0,000218395
0,000222705  Start  0,000222705
0,000227045   0x30  0,000227045
0,000261765   Stop  0,000261765
0,000266095  Start  0,000266095
0,000270435   0x00  0,000270435
0,000305155   Stop  0,000305155
0,000309475  Start  0,000309475
0,000313815   0x00  0,000313815
0,000348535   Stop  0,000348535
0,000352860  Start  0,000352860
0,000357200   0x31  0,000357200
0,000391920   Stop  0,000391920
0,000518775  Start  0,000518775
0,000523115   0x00  0,000523115


In [307]:
df = df[df['Time'] != 'Start']
df = df[df['Time'] != 'Stop']
print(df.head(10))
df = df[4:]
print(df.head(10))

             Time      tiempos
0,000140285  0x00  0,000140285
0,000183675  0x00  0,000183675
0,000227045  0x30  0,000227045
0,000270435  0x00  0,000270435
0,000313815  0x00  0,000313815
0,000357200  0x31  0,000357200
0,000523115  0x00  0,000523115
0,000566495  0x00  0,000566495
0,000609880  0x32  0,000609880
0,000653255  0x00  0,000653255
             Time      tiempos
0,000313815  0x00  0,000313815
0,000357200  0x31  0,000357200
0,000523115  0x00  0,000523115
0,000566495  0x00  0,000566495
0,000609880  0x32  0,000609880
0,000653255  0x00  0,000653255
0,000696650  0x00  0,000696650
0,000740030  0x30  0,000740030
0,000907890  0x00  0,000907890
0,000951270  0x00  0,000951270


In [308]:
df['Time'] = df['Time'].apply(lambda x: int(x, 16))

In [309]:
# Verifica cómo quedó
print(df.head(10))


             Time      tiempos
0,000313815     0  0,000313815
0,000357200    49  0,000357200
0,000523115     0  0,000523115
0,000566495     0  0,000566495
0,000609880    50  0,000609880
0,000653255     0  0,000653255
0,000696650     0  0,000696650
0,000740030    48  0,000740030
0,000907890     0  0,000907890
0,000951270     0  0,000951270


In [310]:
# Crea un diccionario para mapear los valores
mapeo = {
    48: 0,
    49: 1,
    50: 2,
    51: 3,
    52: 4
}

# Aplica el mapeo y pon 'S' en el resto
df['type'] = df['Time'].astype(int).map(mapeo).fillna('S')
df = df.reset_index(drop=True)
print(df.head(10))


   Time      tiempos type
0     0  0,000313815    S
1    49  0,000357200  1.0
2     0  0,000523115    S
3     0  0,000566495    S
4    50  0,000609880  2.0
5     0  0,000653255    S
6     0  0,000696650    S
7    48  0,000740030  0.0
8     0  0,000907890    S
9     0  0,000951270    S


In [311]:
# Inicializamos idx como None
idx = None
# Recorremos los índices donde 'type' es distinto de 'S'
for i in df.index[df['type'] != 'S']:
    # Verificamos que haya al menos dos filas siguientes
    if (i + 2) < len(df):
        # Comprobamos que las dos siguientes filas tengan 'S'
        if (df.loc[i + 1, 'type'] == 'S') and (df.loc[i + 2, 'type'] == 'S'):
            idx = i
            break
    else:
        # Si no hay suficientes filas para verificar, no es un punto válido
        continue

# Si encontramos un índice válido, cortamos el dataframe
if idx is not None:
    df = df.loc[idx:].reset_index(drop=True)
else:
    # Si no se encontró un punto válido, el dataframe queda vacío o como prefieras manejarlo
    df = df.iloc[0:0].reset_index(drop=True)


In [312]:
print(df['type'][0])
print(df['type'][1])
print(df['type'][2])
print(df['type'][3])


1.0
S
S
2.0


In [313]:
# Diccionario para almacenar los arrays
resultados = {0: [], 1: [], 2: [], 3: [], 4: []}
resultados_tiempos = {0: [], 1: [], 2: [], 3: [], 4: []}
# Iterar sobre el dataframe
for i, row in df.iterrows():
    tipo = row['type']
    tipo_tiempo = row['type']  
    if tipo in resultados:
        # Tomar las dos siguientes filas si existen
        sub_df = df.iloc[i+1:i+3]['Time']
        sub_df_times = df.iloc[i+1:i+3]['tiempos']
        # Guardar como array (puedes ajustar qué columnas guardar)
        resultados[tipo].append(sub_df.to_numpy())
        resultados_tiempos[tipo_tiempo].append(sub_df_times.to_numpy())
        




In [314]:

# Opcional: convertir las listas en arrays grandes (si quieres)
import numpy as np
for k in resultados:
    resultados[k] = np.concatenate(resultados[k])
    resultados_tiempos[k] = np.concatenate(resultados_tiempos[k])

# Ahora resultados[0], resultados[1], ... tienen los arrays deseados

In [315]:
print(len(resultados[0]))
print(len(resultados[1]))
print(len(resultados[2]))
print(len(resultados[3]))
print(len(resultados[4]))

6980
6982
6978
32
42


In [316]:
# Creamos un nuevo diccionario con las filas pares eliminadas
resultados_filtrados = {}

for k, arr in resultados_tiempos.items():
    # Tomar los elementos en posiciones impares: 1, 3, 5, ...
    resultados_tiempos[k] = arr[1::2]


In [317]:
arr0 = resultados[0].flatten()
arr1 = resultados[1].flatten()
arr2 = resultados[2].flatten()
arr3 = resultados[3].flatten()
arr4 = resultados[4].flatten()

new_arr0 = []
new_arr1 = []
new_arr2 = []
new_arr3 = []
new_arr4 = []
for i in range(0, len(arr0)-1, 2):
    combined = arr0[i] * 256 + arr0[i+1]
    new_arr0.append(combined)
new_arr0 = np.array(new_arr0)

for i in range(0, len(arr1)-1, 2):
    combined = arr1[i] * 256 + arr1[i+1]
    new_arr1.append(combined)
new_arr1 = np.array(new_arr1)

for i in range(0, len(arr2)-1, 2):
    combined = arr2[i] * 256 + arr2[i+1]
    new_arr2.append(combined)
new_arr2 = np.array(new_arr2)

for i in range(0, len(arr3)-1, 2):
    combined = arr3[i] * 256 + arr3[i+1]
    new_arr3.append(combined)
new_arr3 = np.array(new_arr3)

for i in range(0, len(arr4)-1, 2):
    combined = arr4[i] * 256 + arr4[i+1]
    new_arr4.append(combined)
new_arr4 = np.array(new_arr4)


In [318]:

fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(y=new_arr0, mode='lines', name='Señal 0'))
fig.add_trace(go.Scatter(y=new_arr1, mode='lines', name='Señal 1'))
fig.add_trace(go.Scatter(y=new_arr2, mode='lines', name='Señal 2'))
# fig.add_trace(go.Scatter(y=new_arr3, mode='lines', name='Señal 3'))
# fig.add_trace(go.Scatter(y=new_arr4, mode='lines', name='Señal 4'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified'
)

fig.show()


In [319]:

fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[0], y=new_arr0, mode='lines', name='Señal 0'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified'
)

fig.show()

In [320]:
fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[1], y=new_arr1, mode='lines', name='Señal 1'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified'
)

fig.show()

In [321]:
fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[2], y=new_arr2, mode='lines', name='Señal 2'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified'
)

fig.show()

In [322]:
fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[3], y=new_arr3, mode='lines', name='Señal 3'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified'
)

fig.show()

In [323]:
fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[4], y=new_arr4, mode='lines', name='Señal 4'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified'
)

fig.show()

In [296]:
ceros = np.zeros(100)  
# FFT
X = np.fft.fft(np.concatenate((ceros,(new_arr-32768))))

# Número de muestras
N = len(X)

# Frecuencias asociadas (eje x)
freqs = np.fft.fftfreq(N, 1/f)

# Magnitud (módulo)
magnitud = np.abs(X)

# Para mostrar solo la mitad positiva (frecuencias positivas)
idxs = freqs >= 0

plt.plot(freqs[idxs], magnitud[idxs])
plt.xlabel("Frecuencia (Hz)")
plt.ylabel("Magnitud")
plt.title("Espectro de la señal")
plt.show()

NameError: name 'new_arr' is not defined

In [ ]:

fig = go.Figure()

fig.add_trace(go.Scatter(
    x = freqs[idxs],
    y=magnitud[idxs],
    mode='lines',
    name='Valores concatenados'
))

fig.update_layout(
    title='FFT',
    xaxis_title='Frecuencia',
    yaxis_title='Magnitud',
    hovermode='x unified'
)

fig.show()